## ▶️ Play in Google Colab

Running on Colab? Execute the next cell once — it clones the upstream repo and installs everything. Running locally inside the repo clone? Skip it: the venv already has all of this.

In [ ]:
import os
if not os.path.exists('src/tab_jev'):
    !git clone -q https://github.com/edamame-labs/tab-jev
    get_ipython().run_line_magic('cd', 'tab-jev')
    !pip install -q -e '.[dev]' scikit-learn pytest ipykernel
print('ready:', os.getcwd())

# Exploring **tab-jev** — an in-context learner for text + tabular data

Step-by-step walkthrough of https://github.com/edamame-labs/tab-jev (cloned locally, `uv venv --python 3.12 && uv pip install -e ".[dev]" scikit-learn pytest`).

**The idea in one line:** an LLM judgment layer ("jev") reads the *text* columns and turns them into
typed probabilities; a tabular foundation model ("tab", e.g. TabPFN) does in-context learning on the
labeled rows, seeing both your table columns *and* jev's readings. Jev answers on day one with zero
labels; tab takes over signal-fusion as labels arrive.

**Everything in this notebook runs offline** — the LLM backend is faked (the repo's own test fakes),
so no API keys are needed. Where a real backend would plug in, the notebook says so explicitly.

## 0. Repo layout

| module | role |
|---|---|
| `types.py` | `Question` / `Answer` — the jev request/response contract |
| `backends.py` | the two `Protocol`s: `JevBackend` (text → judgments), `TabBackend` (sklearn-style ICL) |
| `steps.py` | pipeline steps: `JevFeatures`, `JevAnswer`, `TabPredict`, `Blend` |
| `pipeline.py` | `Pipeline` + three ready-made compositions: `jev_then_tab`, `tab_then_jev`, `parallel_blend` |
| `adapters.py` | concrete jev backends: `JevHTTP` (TypeSafe API), `OpenAICompatibleJev` |

In [1]:
import numpy as np, pandas as pd
import tab_jev

print("tab_jev", tab_jev.__version__, "from", tab_jev.__file__)
print("exports:", tab_jev.__all__)

tab_jev 0.1.0.dev0 from /home/hermes/tab-jev/src/tab_jev/__init__.py
exports: ['Answer', 'Blend', 'CachedJev', 'JevAnswer', 'JevBackend', 'JevFeatures', 'JevHTTP', 'OpenAICompatibleJev', 'Pipeline', 'Question', 'Step', 'TabBackend', 'TabLimits', 'TabPredict', 'jev_then_tab', 'jev_to_tabpfn', 'judge_many', 'kev_to_tabicl', 'parallel_blend', 'tab_then_jev']


## 1. The contract: `Question` and `Answer`

A jev question is one of three types:
- `noul` — a yes/no judgment → a single probability (probability of "true")
- `choice` — pick one of named options → a probability distribution
- `score` — 2–10 ordered levels → a distribution over levels

The `Answer` carries the chosen option plus probabilities, and each backend declares its
`probability_source` (native / logprobs / sampling) — how honest the numbers are.

In [2]:
from tab_jev import Question

target = Question("noul", "Will the customer buy?", {"true": "buys", "false": "does not buy"})
print("target:", target.to_dict())

sentiment = Question("choice", "Sentiment of the review?", {"positive": "happy", "negative": "unhappy"})
print("rubric :", sentiment.to_dict())

target: {'type': 'noul', 'instructions': 'Will the customer buy?', 'criteria': {'true': 'buys', 'false': 'does not buy'}}
rubric : {'type': 'choice', 'instructions': 'Sentiment of the review?', 'criteria': {'positive': 'happy', 'negative': 'unhappy'}}


## 2. The two backend protocols (from `backends.py`)

```python
class JevBackend(Protocol):
    probability_source: Literal["native", "logprobs", "sampling"]
    def judge(self, state, questions) -> Mapping[str, Answer]: ...

class TabBackend(Protocol):
    def fit(self, X, y): ...            # labeled context; no training loop
    def predict_proba(self, X): ...     # sklearn-style
```

Any scikit-learn-style classifier is a valid `TabBackend`. The repo ships **fakes** in
`tests/fakes.py` so we can exercise everything offline:

In [3]:
import sys; sys.path.insert(0, "tests")
from fakes import FakeJev, MemorizingTab, LimitedTab, make_data, BUY, SENTIMENT

jev = FakeJev()   # keyword judge: mentions of "good" → 0.8 toward the first option, else 0.2
X, y = make_data(400, seed=0)
X.join(y.rename("bought")).head()

,review,price,bought
0,bad product,20.22,no
1,good product,93.79,no
2,good product,9.48,yes
3,good product,0.49,yes
4,bad product,32.29,no


`make_data` encodes the ground truth: **customers buy when the review is good AND price < 70.**
The *text* carries part of the signal (review quality), the *table* the other part (price) —
exactly the regime tab-jev is built for, where neither plane alone sees the whole rule.

## 3. `jev_then_tab`: the standard composition

`jev_then_tab(target, jev, tab, rubrics=...)` builds:

```
JevFeatures (rubric probabilities → new numeric columns)
  → JevAnswer  (jev answers the target itself, zero-shot)
  → TabPredict (tab ICL-learns on labeled rows from table + jev columns)
```

Key behaviors (straight from `tests/test_pipeline.py`):
- with **no labels**: `.fit()` works, the tab step reports `skipped == {"tab"}`, jev answers alone
- with labels: tab receives columns `["price", "features:review.sentiment=...", "jev:yes", "jev:no"]`
- `jev_columns=[...]` controls which columns jev gets to read (keep the table away from it if you want)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tab_jev import jev_then_tab

pipe = jev_then_tab(BUY, jev, LogisticRegression(), rubrics=SENTIMENT).fit(X, y)
print("tab saw columns:", list(pipe.steps[-1].backend.feature_names_in_))
probs = pipe.predict_proba(X)
print(probs.columns.tolist(), "| rows sum to 1:", bool(np.allclose(probs.sum(axis=1), 1)))
print("AUC (in-sample, fake jev):", round(roc_auc_score((y == "yes"), probs["yes"]), 3))

tab saw columns: ['price', 'features:review.sentiment=positive', 'features:review.sentiment=negative', 'jev:yes', 'jev:no']
['yes', 'no'] | rows sum to 1: True
AUC (in-sample, fake jev): 1.0


## 4. Zero labels: jev answers alone, tab skips gracefully

In [5]:
pipe0 = jev_then_tab(BUY, jev, LogisticRegression(), rubrics=SENTIMENT).fit()
print("skipped steps:", pipe0.skipped)          # {"tab"} — fewer labels than its min
choices = pipe0.predict(X)
print("first 6 answers:", [a.choice for a in choices[:6]])

skipped steps: {'tab'}
first 6 answers: ['no', 'yes', 'yes', 'yes', 'no', 'no']


## 5. The subtle part: **out-of-fold outputs**

If a step that learns from labels feeds another label-using step, feeding its own labeled-row
predictions downstream would leak the label. `Pipeline.fit` therefore computes **out-of-fold**
outputs for such steps: each row's input to the later step was produced *without seeing that row's
label* (5 folds by default).

The `MemorizingTab` fake makes leakage *visible* — certain on rows it was fit on, clueless
elsewhere. If the pipeline leaked, in-sample probabilities would be suspiciously confident:

In [6]:
mem = MemorizingTab()
pipe2 = jev_then_tab(BUY, jev, mem, rubrics=SENTIMENT).fit(X, y)
p_mem = pipe2.predict_proba(X)
print("distinct probability values in-sample:", sorted(set(np.round(p_mem['yes'], 3)))[:6], "...")
print("-> OOF machinery: tab's context columns were built without each row's own label")

distinct probability values in-sample: [0.0, 1.0] ...
-> OOF machinery: tab's context columns were built without each row's own label


## 6. Backend limits: `TabLimits`

Tabular foundation models have context limits (TabPFN caps rows/classes). A backend declares them
via a `limits` attribute, and the pipeline checks *before* calling it — raising a precise error
instead of a confusing server-side failure. (Graceful *skip* is reserved for the too-few-labels
case via `min_labels`; limit violations are loud, by design.)

In [7]:
limited = LimitedTab()   # LogisticRegression with limits = TabLimits(max_rows=10)
try:
    jev_then_tab(BUY, jev, limited, rubrics=SENTIMENT).fit(X, y)   # 400 rows > max_rows=10
except ValueError as e:
    print("rejected loudly:", e)

X10, y10 = make_data(10, seed=1)   # within limits -> fits fine
pipe3 = jev_then_tab(BUY, jev, limited, rubrics=SENTIMENT).fit(X10, y10)
print("fits within limits; skipped:", pipe3.skipped)

rejected loudly: tab backend limits exceeded: 400 rows > 10
fits within limits; skipped: set()


## 7. The other two compositions

- **`tab_then_jev`** — tab predicts first; jev makes the final call with tab's probabilities
  *and labeled examples* in its state (default `shots=8`; those tab predictions are out-of-fold)
- **`parallel_blend`** — jev and tab predict independently; a cross-validated weighted average
  combines them (the `Blend` step learns the weight from labels)

In [8]:
from tab_jev import parallel_blend, tab_then_jev

for name, builder in [("tab_then_jev", tab_then_jev), ("parallel_blend", parallel_blend)]:
    p = builder(BUY, jev, LogisticRegression(), rubrics=SENTIMENT).fit(X, y)
    auc = roc_auc_score((y == "yes"), p.predict_proba(X)["yes"])
    print(f"{name:15s} AUC {auc:.3f}  steps: {[s.name for s in p.steps]}")

tab_then_jev    AUC 0.905  steps: ['features', 'tab', 'jev']


parallel_blend  AUC 1.000  steps: ['features', 'jev', 'tab', 'blend']


## 8. Swapping in the real backends (what you'd change)

The fakes sit exactly where real backends go — nothing else changes:

| slot | fake (used here) | real |
|---|---|---|
| `JevBackend` | `FakeJev` (keyword) | `JevHTTP` (TypeSafe Jev API, key via `TYPESAFE_API_KEY`) or `OpenAICompatibleJev` (any LLM endpoint) |
| `TabBackend` | `LogisticRegression` | `TabPFNClassifier` (`tabpfn-client`) or TabICL (local, `tabicl`) |

```python
from tabpfn_client import TabPFNClassifier
from tab_jev import JevHTTP

pipe = jev_then_tab(BUY, JevHTTP(), TabPFNClassifier(), rubrics=SENTIMENT).fit(X, y)
```

The README's headline numbers (Kickstarter dataset, jev API + TabPFN 3.5): tab-jev beats
table-only TabPFN and jev-text-only calibration at 64/256/1024 labeled rows
(AUC 0.684 → 0.765) — because text judgments and table facts are *complementary*,
the same structure as our toy rule (good review ∧ price < 70).

## 9. Takeaways

1. **Two planes, one contract**: jev = typed text judgments, tab = sklearn-style ICL. Both are `Protocol`s — bring your own.
2. **Degradation paths are explicit**: no labels → tab skips, jev answers; backend context limits → loud, precise `ValueError` before the backend is ever called.
3. **Out-of-fold by default**: the leakage that inflates few-label results is structurally prevented.
4. **Pre-release** (v0.1.0.dev0, API will change) — but `pipeline.py` is ~100 lines; readable in an afternoon.